# SNN SAM: Step-by-step walkthrough

This notebook validates per-timestep Spike Activation Maps (SAM) for SNNs.

- Load config, data, model
- Choose a target spiking layer
- Generate SAMs per time step
- Visualize activations and overlays
- Sanity check spatial support and dynamics

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from src.sam import SpikeActivationMap
import snntorch as snn
from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(2329669718442129658)
config = setup_config_dictionary()
config['DATASET_NAME'] = 'dvs_gesture'
config['MODEL_NAME'] = 'simple_snn'
config['NORM_TYPE'] = 'none'
print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])

In [ ]:
# Load a single sample (batch_size=1)
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
)
events, labels = next(iter(test_loader))  # (B,T,C,H,W) or (B,C,H,W)
if events.ndim == 4:
    events = events.unsqueeze(1)
B, T, C, H, W = events.shape
print('Sample shape:', (B, T, C, H, W))

model = get_model(
    config['MODEL_NAME'],
    num_classes=len(train_loader.dataset.classes),
    num_channels=C,
    time_steps=config['N_TIME_BINS'],
    beta=config['BETA'],
    norm_type=config['NORM_TYPE'],
).to(device)

interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
model.eval()

events = events.to(device)
labels = labels.to(device)
print('Label (gt):', int(labels[0].item()))

## Pick a target spiking layer

We try to detect spiking layers (e.g., `snn.Leaky`) and default to a deep one. If none are found we fall back to the last Conv2d layer.

In [ ]:
# Detect spiking layers (snntorch Leaky, etc.)
spike_layers = [m for m in model.modules() if isinstance(m, snn.Leaky)]
print('Total detected spiking layers:', len(spike_layers))

if len(spike_layers) > 0:
    layer_index = min(6, len(spike_layers) - 1)
    target_layer = spike_layers[layer_index]
    print('Using spiking layer index:', layer_index, '->', target_layer)
else:
    from torch import nn
    conv_layers = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    print('No spiking layers found; falling back to Conv2d layers (count=', len(conv_layers), ')')
    layer_index = min(6, len(conv_layers) - 1)
    target_layer = conv_layers[layer_index]
    print('Using conv layer index:', layer_index, '->', target_layer)

# Create the SAM object
sam = SpikeActivationMap(model, target_layer)
print('SAM initialized. Note: SAM registers forward hooks on the target layer.')

## Forward and prediction

We keep the forward/prediction step for context (SAM is class-agnostic).

In [ ]:
outputs = model(events)  # (B, T, num_classes)
time_mean = outputs.mean(dim=1)
pred = time_mean.argmax(dim=1)
print('Predicted class:', int(pred[0].item()))
print('Time-mean confidence (softmax):', float(time_mean[0].softmax(dim=0).max().item()))

## Generate per-timestep SAMs

SAM is class-agnostic — it uses spikes collected at the target layer across time. You can tune `gamma` to control the recurrence decay used in the NCS computation.

In [ ]:
# Generate SAM maps
cams = sam.generate(
    events,
    gamma=0.5,
    normalize_per_timestep=True,
    upsample_to_input=True,
    return_numpy=False,
)  # (B, T, H, W)
print('SAMs shape:', tuple(cams.shape))

## Visualize

- If input has 2 polarities, combine to grayscale for overlays.
- Plot a time-series of global SAM intensity and overlay CAMs on frames.

In [ ]:
frames = events[0].detach().cpu()  # (T, C, H, W)
if frames.shape[1] == 2:
    fr_np = frames.permute(0, 2, 3, 1).numpy()  # (T,H,W,2)
    comb = combine_polarities(fr_np)  # (T,H,W)
    frames_vis = torch.from_numpy(comb).unsqueeze(1)  # (T,1,H,W)
else:
    frames_vis = frames  # assume RGB or single polarity

stride = max(1, frames_vis.shape[0] // 12)
cts = plot_cam_time_series(
    frames_vis,
    cams[0],
    aggregate='mean',
    frame_stride=stride,
    per_frame_cam_norm=False,
)

## Sanity checks

Show a few sampled SAM frames to confirm spatial support and dynamics.

In [ ]:
Ttot = cams.shape[1]
sel_ts = np.linspace(0, Ttot-1, min(6, Ttot)).astype(int).tolist()
plt.figure(figsize=(3*len(sel_ts), 3))
for i, t in enumerate(sel_ts):
    plt.subplot(1, len(sel_ts), i+1)
    plt.imshow(cams[0, t].detach().cpu(), cmap='jet')
    plt.title(f't={t}')
    plt.axis('off')
plt.tight_layout(); plt.show()

### Cleanup and notes

- Call `sam.remove_hooks()` when you're done to avoid keeping hooks alive.
- Change `gamma` to control how much temporal context previous spikes contribute.
- Set `normalize_per_timestep=False` to view raw NCS magnitudes across time.

In [ ]:
# Remove hooks to be safe
sam.remove_hooks()
print('Hooks removed.')